In [ ]:
import requests

url = "https://api.openalex.org/works"
params = {
    "search": "organic chemistry",
    "per_page": 5,
    "mailto": "youyousquid@163.com"
}
response = requests.get(url, params=params)
data = response.json()

print(type(data))  # 看 data 是什么类型
print(data.keys())  # 看 data 里有哪些键
print(len(data["meta"]))  # 看 results 里有几条论文
print(data["results"][0]) # 打印第一篇论文的完整信息

<class 'dict'>
dict_keys(['meta', 'results', 'group_by'])
7
{'id': 'https://openalex.org/W1825789202', 'doi': None, 'title': "Vogel's Textbook of Practical Organic Chemistry", 'display_name': "Vogel's Textbook of Practical Organic Chemistry", 'relevance_score': 7276.3184, 'publication_year': 2003, 'publication_date': '2003-01-01', 'ids': {'openalex': 'https://openalex.org/W1825789202', 'mag': '1825789202'}, 'language': 'en', 'primary_location': {'id': 'mag:1825789202', 'is_oa': False, 'landing_page_url': 'http://182.160.97.198:8080/xmlui/handle/123456789/386', 'pdf_url': None, 'source': None, 'license': None, 'license_id': None, 'version': None, 'is_accepted': False, 'is_published': False, 'raw_source_name': None, 'raw_type': None}, 'type': 'book', 'indexed_in': [], 'open_access': {'is_oa': False, 'oa_status': 'closed', 'oa_url': None, 'any_repository_has_fulltext': False}, 'authorships': [{'author_position': 'first', 'author': {'id': 'https://openalex.org/A5006158726', 'display_name':

## 2026.09.18
1. 查询条件里的"search"、"per_page"、"mailto"是根据什么来的，是固定用词吗？还是根据提供api的网址的设定来的
   A：都不是固定搭配，参数名是 API 提供方规定的，只能按他们的规则来。关于per_page，因为OpenAlex 的数据量非常大，所以它采用“分页”机制：per_page：每页返回几条；page：你要第几页。有没有“总共 N 条”的参数？ 有的 API 有，但 OpenAlex 没有这个参数。关于"mailto"，OpenAlex 是一个免费开放的学术数据库，它靠“礼貌”维持运转。它建议你在请求里附上邮箱，方便它在必要时联系你，也方便它统计“谁在用我的数据”。不写，它也能工作，只是响应速度可能慢一点，或者在某些极端情况下被限制。不是所有 API 通用的，其他 API 不要求这个参数，就不用写。

2. GET 请求是什么意思，是从目标网址获取数据的意思吗？还有什么别的请求方式 
   A：HTTP 协议（浏览器和服务器通信的规则）里，有几种常见的请求方式：
   请求方式	 作用	     例子
   GET	    获取数据	看一篇论文、搜一个关键词
   POST	    提交数据	提交表单、上传文件、发一条微博
   PUT	    更新数据	修改一篇文章的标题
   DELETE	删除数据	删掉一条记录

3. params=params这种表述我还是没能理解，就像这些天也经常用到这种格式，我不是很能理顺这种逻辑，展开讲讲。
 A：- 等号左边的 params：是 requests.get() 这个函数规定的参数名。你调用它时，必须写 params=...，它才知道“这是查询条件”。
    - 等号右边的 params：是刚才定义的变量名，是一个字典。
    
 params=params 的意思是：把右边这个叫 params 的字典，传给 requests.get() 里那个叫 params 的参数。
 为什么必须写 params= 这个前缀？因为 requests.get() 这个函数有很多参数，比如：requests.get(url, params=..., headers=..., cookies=..., timeout=..., ...) 加上 params=，就是明确告诉 Python：“这个字典，是传给名为 params 的那个参数。” 

4. 为什么 requests.get() 这个函数要专门设一个叫 params 的参数，而不是直接把字典塞进去就行？
   A：函数设计者想让你“把不同的东西分开放”。有几个好处：
   1. 消除歧义，规范化
   2. 可以改变requests.get()里的顺序
   3. 更易读

data（字典）
  └── results（列表）
        └── [0]（字典，一篇论文）
              ├── title（字符串）
              ├── publication_year（整数）
              ├── cited_by_count（整数）
              ├── type（字符串）
              ├── doi（字符串或 None）
              ├── primary_location（字典或 None）
              │     └── source（字典或 None）
              │           └── display_name（期刊名）
              ├── primary_topic（字典或 None）
              │     └── display_name（主题名）
              └── keywords（列表）
                    └── [0]（字典）
                          └── display_name（关键词）

In [20]:
import requests

url = "https://api.openalex.org/works"
params = {
    "search": "organic chemistry",
    "per_page": 200,
    "mailto": "youyousquid@163.com"
}

response = requests.get(url, params=params)
data200 = response.json()

print(data200.keys())
print(len(data200.get("results", [])))

import json

with open("organic_chemistry_works.json", "w") as f:
    json.dump(data200, f, ensure_ascii=False, indent=2)

print("原始数据已保存到 organic_chemistry_works.json")

dict_keys(['meta', 'results', 'group_by'])
200
原始数据已保存到 organic_chemistry_works.json


In [5]:
import json
import pandas as pd

with open("organic_chemistry_works.json", "r", encoding="utf-8") as f:
    data200 = json.load(f)

print(len(data200["results"]))

rows = []
for paper in data200["results"]:
    row = {}
    row["title"] = paper["title"]
    row["publication_year"] = paper["publication_year"]
    row["cited_by_count"] = paper["cited_by_count"]
    journal = (paper.get("primary_location") or {}).get("source") or {}
    journal_name = journal.get("display_name")
    row["journal"] = journal_name
    row["type"] = paper["type"]
    topics = paper.get("primary_topic") or {}
    row["primary_topic"] = topics.get("display_name")
    keywords = [kw["display_name"] for kw in paper.get("keywords", [])]
    row["keywords"] = keywords
    row["doi"] = paper.get("doi")
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("Organic_Chemistry_Papers.csv", index=False)

200


In [ ]:
import requests

url = "https://api.openalex.org/works"
params = {
    "search": "organic chemistry",
    "filter": "from_publication_date:2020-01-01,type:article",
    "sort": "publication_date:desc",
    "per_page": 200,
    "mailto": "youyousquid@163.com"
}

response = requests.get(url, params=params)
data200_latest = response.json()

In [20]:
import json

with open("latest_organic_chemistry_works.json", "w") as f:
    json.dump(data200_latest, f, ensure_ascii=False, indent=2)

print("原始数据已保存到 latest_organic_chemistry_works.json")

import json
import pandas as pd

with open("latest_organic_chemistry_works.json", "r", encoding="utf-8") as f:
    data200_latest = json.load(f)

print(len(data200_latest["results"]))

rows = []
for paper in data200_latest["results"]:
    row = {}
    row["title"] = paper["title"]
    row["publication_year"] = paper["publication_year"]
    row["cited_by_count"] = paper["cited_by_count"]
    journal = (paper.get("primary_location") or {}).get("source") or {}
    journal_name = journal.get("display_name")
    row["journal"] = journal_name
    row["type"] = paper["type"]
    topics = paper.get("primary_topic") or {}
    row["primary_topic"] = topics.get("display_name")
    keywords = [kw["display_name"] for kw in paper.get("keywords", [])]
    row["keywords"] = keywords
    row["doi"] = paper.get("doi")
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("Latest_Organic_Chemistry_Papers.csv", index=False, encoding="utf-8-sig")

原始数据已保存到 latest_organic_chemistry_works.json
200


In [21]:
import pandas as pd
df = pd.read_csv("Latest_Organic_Chemistry_Papers.csv")

print(df.shape)  # 打印数据的形状
print("----")
print(df.columns)  # 打印数据的列名
print("----")
print(df.dtypes)  # 打印数据的类型
print("----")
print(df.head())  # 打印前几行数据
print("----")
print(df.isnull().sum())  # 打印每列缺失值的数量
print("----")
print(df[df["doi"].isnull()])  # 打印缺失 DOI 的行

(200, 8)
----
Index(['title', 'publication_year', 'cited_by_count', 'journal', 'type',
       'primary_topic', 'keywords', 'doi'],
      dtype='str')
----
title                 str
publication_year    int64
cited_by_count      int64
journal               str
type                  str
primary_topic         str
keywords              str
doi                   str
dtype: object
----
                                               title  publication_year  \
0  Hypoxia-responsive supramolecular systems base...              2027   
1  Averrhoa bilimbi L. Juice: An efficient natura...              2026   
2  Probing metabolic integration in obligate, int...              2026   
3  Synergistic enzyme-nanozyme catalysis on a por...              2026   
4      Faculty Authors & Achievers Bibliography 2026              2026   

   cited_by_count                                       journal     type  \
0               0                                        PubMed  article   
1               0    